In [113]:
import re
import polars as pl

In [116]:
ctd_exposure_events = catalog.load("bronze.ctd.ctd_exposure_events")
go_terms = catalog.load("bronze.ontology.go_terms")

[06/11/25 23:53:53] INFO     Loading data from bronze.ctd.ctd_exposure_events             ]8;id=814019;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=372111;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (CSVDataset)...                                                                       

                    INFO     Loading data from bronze.ontology.go_terms (CSVDataset)...   ]8;id=979560;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=148169;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\

In [120]:
go_terms.head()

id,name,type
str,str,str
"""GO_1903199""","""obsolete negative regulation o…","""CLASS"""
"""GO_0032404""","""mismatch repair complex bindin…","""CLASS"""
"""GO_0016534""",null,"""CLASS"""
"""GO_0045762""","""positive regulation of adenyla…","""CLASS"""
"""GO_0102919""","""5,6-dimethylbenzimidazole synt…","""CLASS"""


In [121]:
ctd_exposure_events.head()

exposure_stressor_name,exposure_stressor_id,stressor_source_category,stressor_source_details,number_of_stressor_samples,stressor_notes,number_of_receptors,receptors,receptor_notes,smoking_status,age,age_units_of_measurement,age_qualifier,sex,race,methods,detection_limit,detection_limit_uom,detection_frequency,medium,exposure_marker,exposure_marker_id,marker_level,marker_units_of_measurement,marker_measurement_statistic,assay_notes,study_countries,state_or_province,city_town_region_area,exposure_event_notes,outcome_relationship,disease_name,disease_id,phenotype_name,phenotype_id,phenotype_action_degree_type,anatomy,exposure_outcome_notes,reference,associated_study_titles,enrollment_start_year,enrollment_end_year,study_factors
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""1,1,1-trichloroethane""","""MESH:C024566""","""Commercial product|Dietary|Env…",null,null,null,"""63""","""Children""",null,"""Non-smoker""","""6-10""","""years""","""range""","""Both Males and Females""","""American Indian 3 | Asian …","""gas chromatography mass spectr…","""0.024""","""nanograms per milliliter""","""1.1""","""blood""","""1,1,1-trichloroethane""","""MESH:C024566""","""0.03""","""nanograms per milliliter""","""10th percentile""",null,"""United States""","""Minnesota""","""Minneapolis""","""February 2001""",null,null,null,null,null,null,null,null,"""15743726""","""The School Health Initiative: …","""2001""","""2001""",null
"""1,1,1-trichloroethane""","""MESH:C024566""","""Commercial product|Dietary|Env…",null,null,null,"""63""","""Children""",null,"""Non-smoker""","""6-10""","""years""","""range""","""Both Males and Females""","""American Indian 3 | Asian …","""gas chromatography mass spectr…","""0.024""","""nanograms per milliliter""","""1.1""","""blood""","""1,1,1-trichloroethane""","""MESH:C024566""","""0.03""","""nanograms per milliliter""","""25th percentile""",null,"""United States""","""Minnesota""","""Minneapolis""","""February 2001""",null,null,null,null,null,null,null,null,"""15743726""","""The School Health Initiative: …","""2001""","""2001""",null
"""1,1,1-trichloroethane""","""MESH:C024566""","""Commercial product|Dietary|Env…",null,null,null,"""63""","""Children""",null,"""Non-smoker""","""6-10""","""years""","""range""","""Both Males and Females""","""American Indian 3 | Asian …","""gas chromatography mass spectr…","""0.024""","""nanograms per milliliter""","""1.1""","""blood""","""1,1,1-trichloroethane""","""MESH:C024566""","""0.03""","""nanograms per milliliter""","""50th percentile""",null,"""United States""","""Minnesota""","""Minneapolis""","""February 2001""",null,null,null,null,null,null,null,null,"""15743726""","""The School Health Initiative: …","""2001""","""2001""",null
"""1,1,1-trichloroethane""","""MESH:C024566""","""Commercial product|Dietary|Env…",null,null,null,"""63""","""Children""",null,"""Non-smoker""","""6-10""","""years""","""range""","""Both Males and Females""","""American Indian 3 | Asian …","""gas chromatography mass spectr…","""0.024""","""nanograms per milliliter""","""1.1""","""blood""","""1,1,1-trichloroethane""","""MESH:C024566""","""0.03""","""nanograms per milliliter""","""75th percentile""",null,"""United States""","""Minnesota""","""Minneapolis""","""February 2001""",null,null,null,null,null,null,null,null,"""15743726""","""The School Health Initiative: …","""2001""","""2001""",null
"""1,1,1-trichloroethane""","""MESH:C024566""","""Commercial product|Dietary|Env…",null,null,null,"""63""","""Children""",null,"""Non-smoker""","""6-10""","""years""","""range""","""Both Males and Females""","""American Indian 3 | Asian …","""gas chromatography mass spectr…","""0.024""","""nanograms per milliliter""","""1.1""","""blood""","""1,1,1-trichloroethane""","""MESH:C024566""","""0.04""","""nanograms per milliliter""","""90th percentile""",null,"""United States""","""Minnesota""","""M

In [124]:
df_exp_path = ctd_exposure_events.select(
    ["exposure_stressor_name", "exposure_stressor_id", "phenotype_name", "phenotype_id"]
).filter(pl.col("phenotype_id").is_not_null())
df_exp_path = df_exp_path.with_columns(
    pl.col("phenotype_id").str.replace("GO:", "GO_").alias("phenotype_id")
)
df_exp_path

exposure_stressor_name,exposure_stressor_id,phenotype_name,phenotype_id
str,str,str,str
"""1,12-benzoperylene""","""MESH:C006718""","""cholesterol homeostasis""","""GO_0042632"""
"""1,2,3,4,6,7,8-heptachlorodiben…","""MESH:C046839""","""regulation of blood pressure""","""GO_0008217"""
"""1,2,3,4,6,7,8-heptachlorodiben…","""MESH:C473650""","""regulation of blood pressure""","""GO_0008217"""
"""1,2,3,4,6,7,8-heptachlorodiben…","""MESH:C473650""","""triglyceride metabolic process""","""GO_0006641"""
"""1,2,3,4,7,8-hexachlorodibenzof…","""MESH:C051412""","""regulation of blood pressure""","""GO_0008217"""
…,…,…,…
"""Zinc""","""MESH:D015032""","""glucose homeostasis""","""GO_0042593"""
"""Zinc""","""MESH:D015032""","""glucose homeostasis""","""GO_0042593"""
"""Zinc""","""MESH:D015032""","""glucose metabolic process""","""GO_0006006"""


In [130]:
df_exp_path = df_exp_path.join(
    go_terms, left_on="phenotype_id", right_on="id", how="inner"
)
df_exp_path

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 df_exp_path = df_exp_path.join(go_terms.select(["id", "name"]), left_on="phenotype_id",      │
│   2 df_exp_path                                                                                  │
│   3                                                                                              │
│                                                                                                  │
│ /home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/polars │
│ /_utils/deprecation.py:119 in wrapper                                                            │
│                                                                                                  │
│   116 │   │   │   _rename_keyword_argument(                                                      │
│   117 │   │   │   │   old_name, new_name, kwargs, function.__qualname__, version                 │
│   118 │   │   │   )                                                                              │
│ ❱ 119 │   │   │   return function(*args, **kwargs)                                               │
│   120 │   │                                                                                      │
│   121 │   │   wrapper.__signature__ = inspect.signature(function)  # type: ignore[attr-defined   │
│   122 │   │   return wrapper                                                                     │
│                                                                                                  │
│ /home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/polars │
│ /dataframe/frame.py:7765 in join                                                                 │
│                                                                                                  │
│    7762 │   │   │   │   coalesce=coalesce,                                                       │
│    7763 │   │   │   │   maintain_order=maintain_order,                                           │
│    7764 │   │   │   )                                                                            │
│ ❱  7765 │   │   │   .collect(_eager=True)                                                        │
│    7766 │   │   )                                                                                │
│    7767 │                                                                                        │
│    7768 │   @unstable()                                                                          │
│                                                                                                  │
│ /home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/polars │
│ /_utils/deprecation.py:93 in wrapper                                                             │
│                                                                                                  │
│    90 │   │   │   │                                                                              │
│    91 │   │   │   │   del kwargs["streaming"]                                                    │
│    92 │   │   │                                                                                  │
│ ❱  93 │   │   │   return function(*args, **kwargs)                                               │
│    94 │   │                                                                                      │
│    95 │   │   wrapper.__signature__ = inspect.signature(function)  # type: ignore[attr-defined   │
│    96 │   │   return wrapper                                                                     │
│                                                            

In [126]:
df_exp_path = df_exp_path.rename(
    {
        "exposure_stressor_id": "x_id",
        "exposure_stressor_name": "x_name",
        "phenotype_id": "y_id",
        "phenotype_name": "y_name",
    }
)
df_exp_path

x_name,x_id,y_name,y_id,name
str,str,str,str,str
"""Cadmium""","""MESH:D002104""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…"
"""Cadmium""","""MESH:D002104""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…"
"""Dibutyl Phthalate""","""MESH:D003993""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…"
"""Diethylhexyl Phthalate""","""MESH:D004051""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…"
"""Diethylhexyl Phthalate""","""MESH:D004051""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…"
…,…,…,…,…
"""Tobacco Smoke Pollution""","""MESH:D014028""","""developmental growth""","""GO_0048589""","""developmental growth"""
"""2,4,5,2',4',5'-hexachlorobiphe…","""MESH:C014024""","""IgA receptor activity""","""GO_0019766""","""IgA receptor activity"""
"""Dichlorodiphenyl Dichloroethyl…","""MESH:D003633""","""IgA receptor activity""","""GO_0019766""","""IgA receptor activity"""


In [127]:
df_exp_path = df_exp_path.with_columns(
    [
        pl.lit("exposure").alias("x_type"),
        pl.lit("CTD").alias("x_source"),
        pl.lit("biological process").alias("y_type"),
        pl.lit("GO").alias("y_source"),
        pl.lit("exposure_biological_process").alias("relation"),
        pl.lit("regulates").alias("relation_type"),
    ]
)
df_exp_path

x_name,x_id,y_name,y_id,name,x_type,x_source,y_type,y_source,relation,relation_type
str,str,str,str,str,str,str,str,str,str,str
"""Cadmium""","""MESH:D002104""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""
"""Cadmium""","""MESH:D002104""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""
"""Dibutyl Phthalate""","""MESH:D003993""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""
"""Diethylhexyl Phthalate""","""MESH:D004051""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""
"""Diethylhexyl Phthalate""","""MESH:D004051""","""negative regulation of flagell…","""GO_1901318""","""negative regulation of flagell…","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""
…,…,…,…,…,…,…,…,…,…,…
"""Tobacco Smoke Pollution""","""MESH:D014028""","""developmental growth""","""GO_0048589""","""developmental growth""","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""
"""2,4,5,2',4',5'-hexachlorobiphe…","""MESH:C014024""","""IgA receptor activity""","""GO_0019766""","""IgA receptor activity""","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""
"""Dichlorodiphenyl Dichloroethyl…","""MESH:D003633""","""IgA receptor activity""","""GO_0019766""","""IgA receptor activity""","""exposure""","""CTD""","""biological process""","""GO""","""exposure_biological_process""","""regulates"""


In [128]:
df_exp_path = df_exp_path.select(
    [
        "relation",
        "relation_type",
        "x_id",
        "x_type",
        "x_name",
        "x_source",
        "y_id",
        "y_type",
        "y_name",
        "y_source",
    ]
)
df_exp_path

relation,relation_type,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
str,str,str,str,str,str,str,str,str,str
"""exposure_biological_process""","""regulates""","""MESH:D002104""","""exposure""","""Cadmium""","""CTD""","""GO_1901318""","""biological process""","""negative regulation of flagell…","""GO"""
"""exposure_biological_process""","""regulates""","""MESH:D002104""","""exposure""","""Cadmium""","""CTD""","""GO_1901318""","""biological process""","""negative regulation of flagell…","""GO"""
"""exposure_biological_process""","""regulates""","""MESH:D003993""","""exposure""","""Dibutyl Phthalate""","""CTD""","""GO_1901318""","""biological process""","""negative regulation of flagell…","""GO"""
"""exposure_biological_process""","""regulates""","""MESH:D004051""","""exposure""","""Diethylhexyl Phthalate""","""CTD""","""GO_1901318""","""biological process""","""negative regulation of flagell…","""GO"""
"""exposure_biological_process""","""regulates""","""MESH:D004051""","""exposure""","""Diethylhexyl Phthalate""","""CTD""","""GO_1901318""","""biological process""","""negative regulation of flagell…","""GO"""
…,…,…,…,…,…,…,…,…,…
"""exposure_biological_process""","""regulates""","""MESH:D014028""","""exposure""","""Tobacco Smoke Pollution""","""CTD""","""GO_0048589""","""biological process""","""developmental growth""","""GO"""
"""exposure_biological_process""","""regulates""","""MESH:C014024""","""exposure""","""2,4,5,2',4',5'-hexachlorobiphe…","""CTD""","""GO_0019766""","""biological process""","""IgA receptor activity""","""GO"""
"""exposure_biological_process""","""regulates""","""MESH:D003633""","""exposure""","""Dichlorodiphenyl Dichloroethyl…","""CTD""","""GO_0019766""","""biological process""","""IgA receptor activity""","""GO"""
